In [2]:
import numpy as np
import pandas as pd
from scipy import stats
import sys
sys.path.insert(0, '../')
from ultility.metrics import rolling_std_vol
from ultility.models_garch import garch_forecast_fixed_params
from ultility.models_lstm_baseline import train_lstm_baseline, predict_lstm_baseline_rolling
from ultility.models_transformer import train_transformer
from ultility.lstmgarch import LSTMGARCH
from ultility.transformer_garch import TransformerGARCH
from ultility.eval_vol import save_and_plot
import torch
import torch.nn as nn
import time
import warnings
warnings.filterwarnings('ignore')

In [3]:
datasets = {}
for csv_name, key in [('VN30_INDEX.csv', 'VN30 Index'), ('VN_INDEX.csv', 'VN Index')]:
    df_temp = pd.read_csv(f'../dataset/{csv_name}')
    df_temp['time'] = pd.to_datetime(df_temp['time'], format='mixed', dayfirst=True, errors='coerce')
    df_temp = df_temp.sort_values('time')
    ret_col = 'return_1_day'
    df_temp_filtered = df_temp[df_temp['time'].dt.year >= 2010]
    null_count = df_temp_filtered[ret_col].isna().sum()
    if null_count > 0:
        for idx, val in df_temp_filtered[df_temp_filtered[ret_col].isna()][ret_col].items():
            print(f"{key}: null at {df_temp.loc[idx, 'time'].strftime('%Y-%m-%d')}")
    datasets[key] = df_temp_filtered.set_index('time')[ret_col] * 100

for file in ['DAX_40.csv', 'EuroNext_100.csv', 'IBEX_35.csv', 'KOSPI_index.csv', 'SMI.csv', 'snp500.csv', 'Nikkei_225.csv']:
    df_temp = pd.read_csv(f'../dataset/{file}')
    if 'Date' in df_temp.columns:
        df_temp.rename(columns={'Date': 'time'}, inplace=True)
    df_temp['time'] = pd.to_datetime(df_temp['time'], format='mixed', dayfirst=False, errors='coerce')
    df_temp = df_temp.dropna(subset=['time']).sort_values('time')
    ret_col = 'return_1_day'
    df_temp_filtered = df_temp[df_temp['time'].dt.year >= 2010]
    null_count = df_temp_filtered[ret_col].isna().sum()
    if null_count > 0:
        for idx, val in df_temp_filtered[df_temp_filtered[ret_col].isna()][ret_col].items():
            print(f"{file}: null at {df_temp.loc[idx, 'time'].strftime('%Y-%m-%d')}")
    datasets[file.replace('.csv', '')] = df_temp_filtered.set_index('time')[ret_col] * 100


In [4]:
#Kết quả từ KS split
summary_split = pd.DataFrame({
    "Dataset": [
        "VN30 Index", "VN Index", "DAX_40", "EuroNext_100", "IBEX_35", "KOSPI_index", "SMI", "snp500", "Nikkei_225"
    ],
    "train_size": [1971, 1964, 1983, 2005, 1815, 1944, 1987, 1988, 1677],
    "val_size":   [1096, 1103, 1104, 1115, 1362, 1109, 1135, 1109, 1174],
    "test_size":  [925, 925, 972, 979, 923, 881, 901, 927, 1062],
    "split_i":    [1971, 1964, 1983, 2005, 1815, 1944, 1987, 1988, 1677],
    "split_j":    [3067, 3067, 3087, 3120, 3177, 3053, 3122, 3097, 2851]
})
split_df = summary_split.set_index("Dataset")[["train_size", "val_size", "test_size"]]
split_df.columns = ["Train", "Val", "Test"]
split_df

,Train,Val,Test
Dataset,,,
VN30 Index,1971,1096,925
VN Index,1964,1103,925
DAX_40,1983,1104,972
EuroNext_100,2005,1115,979
IBEX_35,1815,1362,923
KOSPI_index,1944,1109,881
SMI,1987,1135,901
snp500,1988,1109,927
Nikkei_225,1677,1174,1062


In [5]:
def distribution_analysis(data, name):
    returns = pd.Series(data).dropna().values
    nu, loc, scale = stats.t.fit(returns)
    ks_stat, ks_p = stats.kstest(returns, "t", args=(nu, loc, scale))
    norm_loc, norm_scale = stats.norm.fit(returns)
    ks_stat_norm, ks_p_norm = stats.kstest(returns, "norm", args=(norm_loc, norm_scale))
    return nu, loc, scale, ks_stat, ks_p, ks_stat_norm, ks_p_norm



In [7]:
res = []; 
for ds in split_df.index:
    tr, va, te = int(split_df.loc[ds, 'Train']), int(split_df.loc[ds, 'Val']), int(split_df.loc[ds, 'Test'])
    s = datasets[ds].values
    res.append({'Dataset': ds, 'nu_train': distribution_analysis(s[:tr], '')[0], 'nu_val': distribution_analysis(s[tr:tr+va], '')[0], 'nu_test': distribution_analysis(s[tr+va:tr+va+te], '')[0], 'nu_all': distribution_analysis(s[:tr+va+te], '')[0]})
pd.DataFrame(res)

,Dataset,nu_train,nu_val,nu_test,nu_all
0,VN30 Index,4.696197,2.696639,2.385055,3.251474
1,VN Index,4.573257,2.497934,2.509615,3.181560
2,DAX_40,3.424473,2.728275,4.298158,3.232514
3,EuroNext_100,3.502448,2.609397,4.072105,3.203942
4,IBEX_35,4.400113,3.303941,5.962785,3.587792
5,KOSPI_index,3.426984,3.772418,5.169135,3.754324
6,SMI,3.589687,3.226405,4.724666,3.603772
7,snp500,2.705923,2.434559,3.422548,2.715517
8,Nikkei_225,4.926478,3.030612,4.596454,4.053247


In [6]:
seq_len = 60
if not torch.cuda.is_available():
    raise RuntimeError('CUDA is required for this notebook')
torch.backends.cudnn.benchmark = True
device = torch.device('cuda')
predictions = {'GARCH': {}, 'GJR-GARCH': {}, 'LSTM': {}, 'Transformer': {}, 'LSTMGARCH': {}, 'TransformerGARCH': {}}
training_times = {'GARCH': {}, 'GJR-GARCH': {}, 'LSTM': {}, 'Transformer': {}, 'LSTMGARCH': {}, 'TransformerGARCH': {}}

for ds in split_df.index:
    print(f'Dataset: {ds}')
    data = datasets[ds].values if hasattr(datasets[ds], 'values') else np.asarray(datasets[ds])
    tr, va, te = map(int, split_df.loc[ds, ['Train', 'Val', 'Test']])
    train_set, val_set, test_set = data[:tr], data[tr:tr+va], data[tr+va:tr+va+te]
    
    print('  GARCH')
    st = time.time()
    predictions['GARCH'][ds] = garch_forecast_fixed_params(train_set, test_set, 'GARCH')
    training_times['GARCH'][ds] = (time.time() - st) / 60
    
    print('  GJR-GARCH')
    st = time.time()
    predictions['GJR-GARCH'][ds] = garch_forecast_fixed_params(train_set, test_set, 'GJR-GARCH')
    training_times['GJR-GARCH'][ds] = (time.time() - st) / 60
    
    try:
        print('  LSTM')
        st = time.time()
        m, scaler, _, _ = train_lstm_baseline(train_set, val_set, seq_len=seq_len, epochs=50, batch_size=32, device='cuda')
        predictions['LSTM'][ds] = predict_lstm_baseline_rolling(m, train_set, test_set, seq_len, scaler, device='cuda')
        training_times['LSTM'][ds] = (time.time() - st) / 60
    except:
        predictions['LSTM'][ds] = rolling_std_vol(test_set, seq_len)
        training_times['LSTM'][ds] = np.nan
    
    try:
        print('  Transformer')
        st = time.time()
        predictions['Transformer'][ds] = np.abs(train_transformer(train_set, test_set, seq_len=seq_len, epochs=50))
        training_times['Transformer'][ds] = (time.time() - st) / 60
    except:
        predictions['Transformer'][ds] = rolling_std_vol(test_set, seq_len)
        training_times['Transformer'][ds] = np.nan
    
    try:
        print('  LSTMGARCH')
        st = time.time()
        m = LSTMGARCH().to(device)
        opt = torch.optim.Adam(m.parameters(), lr=1e-3)
        X = torch.tensor([train_set[i:i+seq_len] for i in range(len(train_set)-seq_len)], dtype=torch.float32)
        dl = torch.utils.data.DataLoader(X, batch_size=64, shuffle=True, pin_memory=True)
        for _ in range(50):
            for batch in dl:
                opt.zero_grad()
                losses, _ = m(batch.to(device, non_blocking=True))
                losses.backward()
                opt.step()
        m.eval()
        hist = list(train_set)
        preds = []
        with torch.no_grad():
            for x in test_set:
                if len(hist) >= seq_len:
                    seq = torch.tensor(hist[-seq_len:], dtype=torch.float32, device=device).unsqueeze(0)
                    _, sigma2 = m(seq)
                    preds.append(torch.sqrt(sigma2[:, -1]).cpu().numpy()[0])
                hist.append(x)
        predictions['LSTMGARCH'][ds] = np.array(preds)
        training_times['LSTMGARCH'][ds] = (time.time() - st) / 60
    except:
        predictions['LSTMGARCH'][ds] = rolling_std_vol(test_set, seq_len)
        training_times['LSTMGARCH'][ds] = np.nan
    
    try:
        print('  TransformerGARCH')
        st = time.time()
        m = TransformerGARCH().to(device)
        opt = torch.optim.Adam(m.parameters(), lr=1e-3)
        X = torch.tensor([train_set[i:i+seq_len] for i in range(len(train_set)-seq_len)], dtype=torch.float32)
        dl = torch.utils.data.DataLoader(X, batch_size=64, shuffle=True, pin_memory=True)
        for _ in range(50):
            for batch in dl:
                opt.zero_grad()
                losses, _ = m(batch.to(device, non_blocking=True))
                losses.backward()
                opt.step()
        m.eval()
        hist = list(train_set)
        preds = []
        with torch.no_grad():
            for x in test_set:
                if len(hist) >= seq_len:
                    seq = torch.tensor(hist[-seq_len:], dtype=torch.float32, device=device).unsqueeze(0)
                    _, sigma2 = m(seq)
                    preds.append(torch.sqrt(sigma2[:, -1]).cpu().numpy()[0])
                hist.append(x)
        predictions['TransformerGARCH'][ds] = np.array(preds)
        training_times['TransformerGARCH'][ds] = (time.time() - st) / 60
    except:
        predictions['TransformerGARCH'][ds] = rolling_std_vol(test_set, seq_len)
        training_times['TransformerGARCH'][ds] = np.nan

Dataset: VN30 Index
  GARCH
  GJR-GARCH
  LSTM
Epoch [10/50], Train Loss: 0.945038, Val Loss: 1.353267
Epoch [20/50], Train Loss: 0.934027, Val Loss: 1.358675
Epoch [30/50], Train Loss: 0.911496, Val Loss: 1.422322
Epoch [40/50], Train Loss: 0.833458, Val Loss: 1.616308
Epoch [50/50], Train Loss: 0.729363, Val Loss: 1.735276
  Transformer
  LSTMGARCH
  TransformerGARCH
Dataset: VN Index
  GARCH
  GJR-GARCH
  LSTM
Epoch [10/50], Train Loss: 0.947826, Val Loss: 1.212383
Epoch [20/50], Train Loss: 0.941939, Val Loss: 1.215265
Epoch [30/50], Train Loss: 0.907620, Val Loss: 1.275722
Epoch [40/50], Train Loss: 0.870596, Val Loss: 1.326777
Epoch [50/50], Train Loss: 0.799291, Val Loss: 1.426527
  Transformer
  LSTMGARCH
  TransformerGARCH
Dataset: DAX_40
  GARCH
  GJR-GARCH
  LSTM
Epoch [10/50], Train Loss: 1.010833, Val Loss: 1.150714
Epoch [20/50], Train Loss: 0.998377, Val Loss: 1.171576
Epoch [30/50], Train Loss: 0.957067, Val Loss: 1.274899
Epoch [40/50], Train Loss: 0.910636, Val Loss: 

In [ ]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

2.7.1+cu118
True
NVIDIA GeForce RTX 3060 Laptop GPU
